# V48 — El injerto en las condiciones reales del cuello

**Qué cambia respecto de V47 (tres decisiones).**
1. **Presión perivascular P_a** (Masson 2011): el tejido del cuello aprieta la carótida desde afuera, P_a = a·exp(b·r_m/r_m,d). La sangre empuja con P_i; la pared soporta solo P_i − P_a. El radio diastólico r_m,d sale del modelo con la **presión diastólica media del grupo** (NT 64,2 mmHg; HT 72,6 mmHg; Masson, Tabla 1).
2. El injerto, en el mismo lugar, recibe **la misma P_a** que la arteria a cada presión de la sangre.
3. Se copia el **diámetro de la luz** (el que ve la sangre), en mm. El injerto suma una variable de diseño: su **radio interior de fabricación R_g** (el del mandril).

Además, la pared de la arteria es **incompresible**: su espesor cambia con la presión (en diástole vale el IMT de Engelen 2013, con sus ecuaciones exactas).

**El procedimiento (igual que V47, 3 pasos).** 1 arranque (A: MCO lineal de pared delgada · B: red) → 2 MCO no lineal (reales) → 3 láminas enteras de 0,025 mm con minimax contra la curva del paso 2. El diseño entero se compara siempre con la **curva original**.

**Celdas.** 0 configuración · 1 sujetos (P_a, luz, pared incompresible) · 2 arterias sintéticas · 3 injerto con R_g · 4 ajuste por mínimos cuadrados · 5 dataset con etiquetas · 6 partición · 7 red (se entrena de nuevo: entrada 282, salida N+2) · **8 los dos caminos con curvas limpias** · **9 figuras** · **10 curvas medidas**. La presión va siempre en la ordenada: kPa a la izquierda, mmHg a la derecha; abajo, escala relativa P/13,332 kPa. La abscisa es el diámetro de la luz en mm.

## Celda 0 — Configuración (ejecutar siempre primero)

In [ ]:
# CELDA 0: CONFIGURACIÓN (ejecutar siempre primero)
#   V48 = V47 con la arteria y el injerto puestos en las condiciones REALES del cuello:
#     1) Presión perivascular P_a (Masson 2011): el tejido que rodea a la carótida la aprieta desde afuera,
#        P_a = a·exp(b·r_m/r_m,d). La sangre empuja con P_i; la pared soporta solo P_i − P_a.
#        r_m,d (radio en diástole) se obtiene del modelo con la presión diastólica MEDIA del grupo (Masson, Tabla 1, presión braquial):
#        NT 64,2 mmHg, HT 72,6 mmHg.
#     2) El injerto, implantado en el mismo lugar, recibe la MISMA P_a que la arteria a cada presión de la sangre P_i.
#     3) Lo que se copia es el DIÁMETRO DE LA LUZ (el que ve la sangre), en milímetros: el injerto tiene además su propio
#        radio interior de fabricación R_g (una variable más del diseño, como el diámetro del mandril).
#     + La pared de la arteria es incompresible: su espesor cambia con la presión (h = IMT de Engelen 2013 en diástole).
#   Procedimiento (igual que V47): 1 arranque (A: MCO lineal de pared delgada · B: red) → 2 MCO no lineal (reales) →
#   3 láminas enteras de 0,025 mm (minimax contra la curva del paso 2). El diseño entero se compara con la curva ORIGINAL.
import os
import numpy as np

N_BASES = 5                      # ← número de materiales: 5 o 6
VERSION = f"v48_{N_BASES}"
SEED = 42
DIR_MASSON = "."
ENTRENAR = "auto"                # "auto": carga la red si sus archivos están; si no, la entrena

# Presión de la sangre P_i (la misma ventana para todos) y conversión
MMHG = 0.133322                  # kPa por mmHg
P_MIN_MMHG, P_MAX_MMHG, N_P = 60.0, 200.0, 141
P_GRID = np.linspace(P_MIN_MMHG, P_MAX_MMHG, N_P) * MMHG          # P_i [kPa]
P_REF_KPA = 13.332               # escala relativa: P/P_ref, con P_ref = 13,332 kPa (= 100 mmHg)
X_FINO = np.linspace(0.85, 1.45, 1201)                              # λθ en r_m donde se calcula la presión antes de invertir

# Presión diastólica media de cada grupo (Masson et al. 2011, Tabla 1) [mmHg]
PAD_GRUPO = {"NT": 64.2, "HT": 72.6}

# Geometría documentada
#   R_m, Θ0, λz: Masson et al. (2011), Tablas 2 y 4 (por sujeto). R_m = radio media–adventicia de la configuración sin tensiones.
#   Espesor íntima–media en diástole: Engelen et al. (2013), Eur Heart J 34:2368–2380, ecuaciones de la mediana de referencia:
#   mujeres 321,7 + 4,971·edad µm; hombres 323,5 + 5,201·edad µm.
IMT_ENGELEN = {"F": (321.7, 4.971), "M": (323.5, 5.201)}
N_RADIAL = 60

# Materiales: nudos (V41) y módulos E_j [kPa por unidad de λ] (los mismos de V43–V47)
MATERIALES = {6: dict(nudos=[0.8193, 1.0227, 1.0774, 1.1099, 1.1318, 1.1478],
                      E=[451.9565, 1632.2251, 1530.1154, 1769.7480, 3289.3120, 4965.9454]),
              5: dict(nudos=[0.8228, 1.0243, 1.0881, 1.1226, 1.1447],
                      E=[522.2327, 1759.0928, 1904.0067, 3923.1919, 5916.2233])}
H_CAPA_MAX = 3.0                 # cota física de búsqueda de una capa [mm] (no es una restricción de diseño)
R_G_LIM = (1.0, 5.0)             # cota física de búsqueda del radio interior de fabricación R_g [mm]
N_GAUSS_CAPA = 8

ERROR_MAX_PCT = 3.0
ESP_LAMINA_MM = 0.025            # espesor de una lámina [mm]: cada capa = n láminas enteras (n ≥ 0)
FRAC_SUJETOS_TEST = 0.25
ETQ_P, ETQ_D = r"$P_i$ (presión de la sangre) [kPa]", r"$D_i$ (diámetro de la luz) [mm]"

_v = VERSION
ARCHIVO_MODELO, ARCHIVO_SCALERS, ARCHIVO_HISTORIAL = (f"modelo_rn_{_v}.keras", f"scalers_{_v}.pkl", f"historial_entrenamiento_{_v}.csv")
ARCHIVO_ETIQUETAS = f"etiquetas_{_v}.npz"
if ENTRENAR == "auto":
    ENTRENAR = not all(os.path.exists(f) for f in (ARCHIVO_MODELO, ARCHIVO_SCALERS, ARCHIVO_HISTORIAL))
print(f"✅ {VERSION} | {N_BASES} materiales | se copia el diámetro de la luz D_i(P_i) para P_i de {P_MIN_MMHG:g} a {P_MAX_MMHG:g} mmHg, "
      f"con presión perivascular | red: {'se entrena' if ENTRENAR else 'se carga de ' + ARCHIVO_MODELO}")

## Celda 1 — Los 41 sujetos: presión perivascular, pared incompresible y curva de la luz

In [ ]:
# CELDA 1: LOS 41 SUJETOS — ley de Masson, pared incompresible, presión perivascular y curva de la luz r_i(P_i)
#   Dibujo: un anillo (la pared íntima–media) entre la luz (radio r_i) y la adventicia (radio r_m). La sangre empuja desde adentro
#   con P_i; el tejido del cuello aprieta desde afuera con P_a. La pared soporta la diferencia:
#       P_i − P_a = ∫ (σθθ − σrr)/r dr  (de r_i a r_m)          P_a = a·exp(b·r_m/r_m,d)   (Masson 2011, ecs. 11 y 12)
#   Pared incompresible (Masson, ec. 3): R² = R_m² − (π·λz/Θ0)·(r_m² − r²). El radio interior sin tensiones R_i queda fijo y se
#   obtiene del estado diastólico: en diástole la pared mide h = IMT (Engelen 2013) y P_i = PAD media del grupo.
#   Como r_m,d depende de R_i y R_i de r_m,d, se resuelve por punto fijo (converge en pocas vueltas).
#   Resultado de cada sujeto: y = [r_i(P_i) en los 141 valores de P_i ; P_w(P_i) = P_i − P_a en esos mismos valores].
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq

COLS_PARAM = ["Rm_mm", "Theta0_deg", "lambda_ax", "c_kPa", "c1_kPa", "c2", "c1circ_kPa", "c2circ", "alpha_deg",
              "Tm_kPa", "lambda0", "lambda_m", "a_kPa", "b"]


def imt_mm(edad, sexo):
    """Mediana de referencia del IMT de carótida común [mm] (Engelen et al. 2013): 321,7 + 4,971·edad (F); 323,5 + 5,201·edad (M) µm."""
    a0, a1 = IMT_ENGELEN[sexo]
    return (a0 + a1 * edad) / 1000.0


def sigma_masson(lt, p):
    """σθθ − σrr (λθ) [kPa]: elastina neo-Hookeana + 4 familias de colágeno + tono activo (Masson et al. 2011)."""
    lt = np.asarray(lt, float)
    lz, al = p["lambda_ax"], np.deg2rad(p["alpha_deg"])
    lam_d = np.sqrt(lt ** 2 * np.sin(al) ** 2 + lz ** 2 * np.cos(al) ** 2)
    suma = np.zeros_like(lt)
    for lam_k, c1, c2, ang in [(lz * np.ones_like(lt), p["c1_kPa"], p["c2"], 0.0), (lt, p["c1circ_kPa"], p["c2circ"], np.pi / 2),
                               (lam_d, p["c1_kPa"], p["c2"], al), (lam_d, p["c1_kPa"], p["c2"], -al)]:
        E = lam_k ** 2 - 1.0
        suma = suma + c1 * E * np.exp(c2 * E ** 2) * np.sin(ang) ** 2
    activo = p["Tm_kPa"] * lt * (1.0 - ((p["lambda_m"] - lt) / (p["lambda_m"] - p["lambda0"])) ** 2)
    return -p["c_kPa"] * (1.0 / (lt * lz)) ** 2 + p["c_kPa"] * lt ** 2 + lt ** 2 * suma + activo


def _radios(lt_m, p, Ri):
    """r_m y r_i en el cuerpo [mm] para λθ en r_m, con el radio interior sin tensiones Ri (incompresible, Masson ec. 3)."""
    Th, lz, Rm = np.deg2rad(p["Theta0_deg"]), p["lambda_ax"], p["Rm_mm"]
    rm = np.asarray(lt_m, float) * Th * Rm / np.pi
    ri = np.sqrt(np.maximum(rm ** 2 - (Th / (np.pi * lz)) * (Rm ** 2 - Ri ** 2), 1e-12))
    return rm, ri


def presion_pared(lt_m, p, Ri):
    """P_i − P_a [kPa] = ∫ (σθθ − σrr)/r dr de r_i a r_m, para λθ en r_m."""
    Th, lz, Rm = np.deg2rad(p["Theta0_deg"]), p["lambda_ax"], p["Rm_mm"]
    rm, ri = _radios(lt_m, p, Ri)
    s = np.linspace(0.0, 1.0, N_RADIAL)[None, :]
    rr = ri[:, None] + (rm - ri)[:, None] * s
    R = np.sqrt(np.maximum(Rm ** 2 - (np.pi * lz / Th) * (rm[:, None] ** 2 - rr ** 2), 1e-12))
    return np.trapezoid(sigma_masson(np.pi * rr / (Th * R), p) / rr, rr, axis=1)


def _Ri_de_diastole(lt_d, p, h):
    """Radio interior sin tensiones tal que en diástole (λθ = lt_d en r_m) la pared mida h."""
    Th, lz, Rm = np.deg2rad(p["Theta0_deg"]), p["lambda_ax"], p["Rm_mm"]
    rmd = lt_d * Th * Rm / np.pi
    return np.sqrt(max(Rm ** 2 - (np.pi * lz / Th) * (rmd ** 2 - (rmd - h) ** 2), 1e-6))


def estado_arteria(p, h, pad_mmhg):
    """Punto fijo del estado diastólico: devuelve (λθ,d en r_m, R_i) o None."""
    Pd, Pa_d, ld = pad_mmhg * MMHG, p["a_kPa"] * np.exp(p["b"]), 1.10
    if Pa_d >= Pd:
        return None
    for _ in range(30):
        Ri = _Ri_de_diastole(ld, p, h)
        f = lambda l: presion_pared(np.array([l]), p, Ri)[0] + Pa_d - Pd
        try:
            ld_n = brentq(f, X_FINO[0], X_FINO[-1], xtol=1e-8)
        except ValueError:
            return None
        if abs(ld_n - ld) < 1e-7:
            return ld_n, _Ri_de_diastole(ld_n, p, h)
        ld = ld_n
    return None


def curva_arteria(p, h, pad_mmhg, todo=False):
    """y = [r_i(P_i) ; P_w(P_i)] en P_GRID (2·N_P valores), o None si la curva no cubre la ventana o no es creciente."""
    with np.errstate(all="ignore"):
        est = estado_arteria(p, h, pad_mmhg)
        if est is None:
            return None
        ld, Ri = est
        Pw = presion_pared(X_FINO, p, Ri)
        Pa = p["a_kPa"] * np.exp(p["b"] * X_FINO / ld)
        Pi = Pw + Pa
        rm, ri = _radios(X_FINO, p, Ri)
    if not (np.all(np.isfinite(Pi)) and Pi[0] <= P_GRID[0] and Pi[-1] >= P_GRID[-1]):
        return None
    k0 = np.searchsorted(Pi, P_GRID[0], side="right") - 1         # tramo que se usa: debe ser creciente
    k1 = np.searchsorted(Pi, P_GRID[-1]) + 1
    if np.any(np.diff(Pi[k0:k1 + 1]) <= 0) or np.any(np.diff(Pw[k0:k1 + 1]) <= 0) or np.any(np.diff(ri[k0:k1 + 1]) <= 0):
        return None
    lam = np.interp(P_GRID, Pi[k0:k1 + 1], X_FINO[k0:k1 + 1])
    y = np.r_[np.interp(lam, X_FINO, ri), np.interp(lam, X_FINO, Pw)]
    if todo:
        return y, dict(lam=lam, ld=ld, Ri=Ri, rm=np.interp(lam, X_FINO, rm), Pa=P_GRID - y[N_P:], h=np.interp(lam, X_FINO, rm - ri))
    return y


def partes(y):
    """(r_i [mm], P_w [kPa]) de un vector y."""
    y = np.asarray(y, float)
    return y[:N_P], y[N_P:]


POB = pd.concat([pd.read_csv(f"{DIR_MASSON}/{f}").assign(grupo=g) for f, g in
                 [("tabla2_masson_NT.csv", "NT"), ("tabla4_masson_HT.csv", "HT")]], ignore_index=True)
POB["id"] = POB["grupo"] + POB["subject"].map("{:02d}".format)
POB["h_mm"] = [imt_mm(e, s) for e, s in zip(POB["age"], POB["gender"])]
POB["PAD_mmHg"] = POB["grupo"].map(PAD_GRUPO)
_res = [curva_arteria(r, r["h_mm"], r["PAD_mmHg"], todo=True) for _, r in POB.iterrows()]
assert all(v is not None for v in _res), "algún sujeto no cubre la ventana de presión"
POB_Y = np.array([v[0] for v in _res])                    # la curva objetivo de cada sujeto: [r_i ; P_w]
POB_INFO = [v[1] for v in _res]
POB_RI, POB_PW = POB_Y[:, :N_P], POB_Y[:, N_P:]
_k = lambda mm: int(round((mm - P_MIN_MMHG) / (P_MAX_MMHG - P_MIN_MMHG) * (N_P - 1)))
POB["Pa_dias_mmHg"] = [(r["a_kPa"] * np.exp(r["b"])) / MMHG for _, r in POB.iterrows()]
POB["Pa_120_mmHg"] = [v["Pa"][_k(120)] / MMHG for v in POB_INFO]
POB["D_60_mm"], POB["D_120_mm"], POB["D_200_mm"] = 2 * POB_RI[:, 0], 2 * POB_RI[:, _k(120)], 2 * POB_RI[:, -1]
POB["h_120_mm"] = [v["h"][_k(120)] for v in POB_INFO]
_rng = np.random.default_rng(SEED)
POB["test"] = False
for g in ("NT", "HT"):
    i_g = POB.index[POB["grupo"] == g].to_numpy()
    POB.loc[_rng.choice(i_g, size=int(round(FRAC_SUJETOS_TEST * len(i_g))), replace=False), "test"] = True
MASK_TEST = POB["test"].to_numpy()
MASK_TRAIN = ~MASK_TEST
print(f"✅ 41 sujetos | IMT en diástole (Engelen 2013): {POB.h_mm.min():.3f}–{POB.h_mm.max():.3f} mm | P_a en diástole: "
      f"{POB.Pa_dias_mmHg.min():.0f}–{POB.Pa_dias_mmHg.max():.0f} mmHg (mediana {POB.Pa_dias_mmHg.median():.0f}) | "
      f"diámetro de la luz a 120 mmHg: {POB.D_120_mm.min():.2f}–{POB.D_120_mm.max():.2f} mm")
print(f"   test: {', '.join(POB.id[MASK_TEST])}")
print(POB[["id", "age", "gender", "PAD_mmHg", "h_mm", "h_120_mm", "Pa_dias_mmHg", "Pa_120_mmHg", "D_60_mm", "D_120_mm", "D_200_mm"]]
      .round(2).to_string(index=False))


def eje_mmhg(ax, escala=1.0):
    """Ordenada derecha en mmHg, equivalente a la izquierda (P [kPa] si escala = 1; P/P_ref si escala = P_REF_KPA)."""
    a2 = ax.secondary_yaxis("right", functions=(lambda v: v * escala / MMHG, lambda v: v * MMHG / escala))
    a2.set_ylabel("P [mmHg]")
    return a2


def figura_sujetos(escala, etq_izq, archivo, titulo):
    """41 sujetos: presión de la sangre P_i (llena) y la que soporta la pared P_i − P_a (punteada) en función del diámetro de la luz."""
    fig, ax = plt.subplots(figsize=(11, 6.5))
    ax.axhspan(P_GRID[0] / escala, P_GRID[-1] / escala, color="tab:green", alpha=0.08, label="Ventana 60–200 mmHg de P_i")
    for g, c in (("NT", "tab:blue"), ("HT", "tab:red")):
        m = (POB.grupo == g).to_numpy()
        for i in np.flatnonzero(m):
            ax.plot(2 * POB_RI[i], P_GRID / escala, color=c, lw=1.6)
            ax.plot(2 * POB_RI[i], POB_PW[i] / escala, color=c, lw=0.8, ls=":", alpha=0.7)
        ax.plot([], [], color=c, lw=1.6, label=f"{g} (n = {m.sum()}): P_i, presión de la sangre")
        ax.plot([], [], color=c, lw=0.8, ls=":", label=f"{g}: P_i − P_a, lo que soporta la pared")
    ax.set(xlabel=ETQ_D, ylabel=etq_izq, title=titulo, ylim=(0, None))
    eje_mmhg(ax, escala)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8.5, loc="upper left")
    fig.tight_layout()
    plt.savefig(archivo, dpi=180, bbox_inches="tight")
    plt.show()


figura_sujetos(1.0, ETQ_P, f"sujetos_P_kPa_{VERSION}.png", "Los 41 sujetos: presión en función del diámetro de la luz")
figura_sujetos(P_REF_KPA, f"P / P_ref  (P_ref = {P_REF_KPA:.3f} kPa)", f"sujetos_P_relativa_{VERSION}.png",
               f"Los 41 sujetos: presión relativa a {P_REF_KPA:.3f} kPa (= 100 mmHg)")

## Celda 2 — Arterias sintéticas (espacio de trabajo)

In [ ]:
# CELDA 2: ESPACIO DE TRABAJO — arterias sintéticas (ley de Masson con parámetros de los sujetos de entrenamiento, mezclados y perturbados)
#   2 constitutiva: mezcla de 2–3 sujetos + perturbación chica   3 amplia: 2–4 sujetos, perturbación grande y extrapolación
#   5 uniforme: cada parámetro al azar entre el mínimo y el máximo de los sujetos   6 pacientes: los 31 de entrenamiento
#   Se mezclan también el IMT (perturbado un 10 %) y los parámetros a, b de la presión perivascular; la presión diastólica se sortea
#   entre 60 y 80 mmHg. Se acepta la curva si cubre la ventana 60–200 mmHg y es creciente.
import numpy as np
import pandas as pd

N_FAM = {2: 2000, 3: 1250, 5: 750, 6: 1000}
FAMILIAS = {1: "injertos", 2: "constitutiva", 3: "constitutiva_amplia", 5: "constitutiva_uniforme", 6: "pacientes"}
PERTURB = {2: dict(k=(2, 3), sd_mult=0.10, sd_adit=0.15, gamma=(1.0, 1.0)), 3: dict(k=(2, 4), sd_mult=0.20, sd_adit=0.30, gamma=(1.0, 1.35))}
PAD_SINT = (60.0, 80.0)
PARAM_MULT = ["Rm_mm", "c_kPa", "c1_kPa", "c1circ_kPa", "Tm_kPa", "a_kPa", "h_mm"]
COLS = COLS_PARAM + ["h_mm"]
PARAM_ADIT = [c for c in COLS if c not in PARAM_MULT]
rng = np.random.default_rng(2026)
TH = POB.loc[MASK_TRAIN, COLS].to_numpy(float)
SD = POB[COLS].to_numpy(float).std(axis=0)
ix = {c: i for i, c in enumerate(COLS)}
iM, iA = [ix[c] for c in PARAM_MULT], [ix[c] for c in PARAM_ADIT]
es_mult = np.isin(np.arange(len(COLS)), iM)
a_z = lambda t: np.where(es_mult, np.log(np.maximum(t, 1e-9)), t)
de_z = lambda z: np.where(es_mult, np.exp(np.where(es_mult, z, 0.0)), z)
Z = a_z(TH)


def curva(th):
    """Curva y = [r_i ; P_w] de una arteria con parámetros th y una presión diastólica sorteada, o None si no es válida."""
    if th[ix["lambda_m"]] <= th[ix["lambda0"]] + 0.2:
        return None
    try:
        return curva_arteria(dict(zip(COLS, th)), th[ix["h_mm"]], rng.uniform(*PAD_SINT))
    except (FloatingPointError, ValueError):
        return None


def sortear(f):
    if f == 5:
        return curva(de_z(rng.uniform(a_z(TH.min(0)), a_z(TH.max(0)))))
    cfg = PERTURB[f]
    k = rng.integers(cfg["k"][0], cfg["k"][1] + 1)
    idx = rng.choice(len(TH), size=k, replace=False)
    w = rng.dirichlet(np.ones(k))
    z = Z.mean(0) + rng.uniform(*cfg["gamma"]) * (w @ Z[idx] - Z.mean(0))
    z[iM] += rng.normal(0.0, cfg["sd_mult"], size=len(iM))
    z[iA] += rng.normal(0.0, cfg["sd_adit"], size=len(iA)) * SD[iA]
    th = de_z(z)
    th[ix["alpha_deg"]] = np.clip(th[ix["alpha_deg"]], 5.0, 85.0)
    th[ix["lambda_ax"]] = max(th[ix["lambda_ax"]], 1.0)
    th[[ix["c2"], ix["c2circ"]]] = np.maximum(th[[ix["c2"], ix["c2circ"]]], 0.5)
    th[ix["Theta0_deg"]] = np.clip(th[ix["Theta0_deg"]], 60.0, 300.0)
    th[ix["h_mm"]] = np.clip(th[ix["h_mm"]], 0.35, 0.90)
    th[ix["b"]] = np.clip(th[ix["b"]], 0.0, 5.0)
    return curva(th)


LIMPIAS, ACEPTACION = {}, {}
np.seterr(all="ignore")
for f in (2, 3, 5):
    Y, intentos = [], 0
    while len(Y) < N_FAM[f]:
        y, intentos = sortear(f), intentos + 1
        if y is not None:
            Y.append(y)
    LIMPIAS[f], ACEPTACION[f] = np.array(Y), N_FAM[f] / intentos
W_ESP = np.vstack([LIMPIAS[f] for f in (2, 3, 5)])
print(f"✅ Espacio de trabajo: {len(W_ESP)} arterias sintéticas | aceptación: "
      + ", ".join(f"{FAMILIAS[f]} {100 * ACEPTACION[f]:.1f}%" for f in (2, 3, 5))
      + f" | diámetro de la luz a 60 mmHg: {2 * W_ESP[:, 0].min():.2f}–{2 * W_ESP[:, 0].max():.2f} mm | "
        f"a 200 mmHg: {2 * W_ESP[:, N_P - 1].min():.2f}–{2 * W_ESP[:, N_P - 1].max():.2f} mm | "
        f"P_i − P_a a 120 mmHg: {W_ESP[:, N_P + 60].min() / MMHG:.0f}–{W_ESP[:, N_P + 60].max() / MMHG:.0f} mmHg")

## Celda 3 — El injerto: pared gruesa, rectas truncadas, preestiramiento δ y radio R_g

In [ ]:
# CELDA 3: EL INJERTO — tubo cerrado de pared gruesa, una capa por material (rectas truncadas), preestiramiento δ y radio R_g
#   Dibujo: el injerto recién fabricado sobre el mandril es un tubo de radio interior R_g con capas pegadas una sobre otra
#   (la de menor nudo junto a la sangre). Implantado, la sangre empuja con P_i y el cuello aprieta con la misma P_a que a la
#   arteria: la pared del injerto soporta P_w = P_i − P_a.
#   Material j: σθθ − σrr = E_j·max(0, λ − κ_j), λ = r/R el estiramiento circunferencial local.
#   Diseño θ = (h_1 … h_N, δ, R_g): espesores [mm], preestiramiento común δ (cada capa engancha cuando la pared media llega a
#   k_j + δ: rizado de las fibras al fabricar) y radio interior de fabricación R_g [mm] (el diámetro del mandril).
#   Incompresible, deformación plana: r² − r_i² = R² − R_g².  P_w = Σ_j ∫_capa j σ_j/r dr.
#   Se compara la luz: r_i del injerto bajo P_w(P_i) contra r_i de la arteria a la misma P_i.
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar

NUDOS = np.asarray(MATERIALES[N_BASES]["nudos"], float)
E_MAT = np.asarray(MATERIALES[N_BASES]["E"], float)
ORDEN = np.argsort(NUDOS)
_GX, _GW = np.polynomial.legendre.leggauss(N_GAUSS_CAPA)
DELTA_MAX = 0.25
X_INJ = np.linspace(0.80, 1.45, 651)          # λ de la pared media del injerto donde se calcula P_w antes de invertir


def aportes(h, d, Rg, x):
    """P_w [kPa] que aporta cada capa por mm de espesor, para λ de pared media x (una columna por material)."""
    hh = np.maximum(np.asarray(h, float), 1e-6)
    hs = hh[ORDEN]
    Rb = Rg + np.concatenate([[0.0], np.cumsum(hs)])
    Rmed = Rg + hs.sum() / 2
    ri2 = (np.asarray(x) * Rmed) ** 2 - (Rmed ** 2 - Rg ** 2)
    B = np.zeros((len(x), N_BASES))
    for pos, j in enumerate(ORDEN):
        a, b = Rb[pos], Rb[pos + 1]
        Rc = (a + b) / 2
        kap = np.sqrt(max(((NUDOS[j] + d) * Rmed) ** 2 - (Rmed ** 2 - Rg ** 2) + Rc ** 2 - Rg ** 2, 1e-12)) / Rc
        R = Rc + (b - a) / 2 * _GX
        w = (b - a) / 2 * _GW
        r = np.sqrt(np.maximum(ri2[:, None] + R[None, :] ** 2 - Rg ** 2, 1e-12))
        B[:, j] = (E_MAT[j] * np.maximum(0.0, r / R[None, :] - kap) * R[None, :] / r ** 2 * w[None, :]).sum(1) / hh[j]
    return B, np.sqrt(np.maximum(ri2, 1e-12))


def curva_injerto(th, Pw):
    """r_i [mm] del injerto con diseño th = (h_1 … h_N, δ, R_g) bajo las cargas de pared Pw [kPa]."""
    th = np.asarray(th, float)
    h = th[:N_BASES]
    B, ri = aportes(h, th[N_BASES], th[N_BASES + 1], X_INJ)
    P = np.maximum.accumulate(B @ np.maximum(h, 0.0))
    k = np.flatnonzero(np.diff(P) > 1e-9)
    if len(k) < 2:
        return np.full(len(Pw), ri[0])
    return np.interp(Pw, P[k + 1], ri[k + 1], left=ri[0], right=ri[-1])


def errores(y, ri):
    """(E_rms, e_máx) de la luz, en % del rango de r_i de la curva objetivo y (de 60 a 200 mmHg)."""
    r, d = partes(y)[0] - np.asarray(ri), np.ptp(partes(y)[0])
    return 100 * np.sqrt(np.mean(r ** 2)) / d, 100 * np.abs(r).max() / d


def curva_de(th, y):
    """Luz del injerto th bajo las cargas de pared de la curva objetivo y."""
    return curva_injerto(th, partes(y)[1])


def delgada(y, d, R0, H=0.5):
    """Pared delgada (Laplace, P = σ·t/r con t = h/λ el espesor en el cuerpo): P_w ≈ Σ c_j·max(0, λ − k_j − δ)/λ², c_j = E_j·h_j/R0,
    λ = r/R0, con r el radio de la pared media en el cuerpo: r² = r_i² + R0² − R_g², R_g = R0 − H/2 (incompresible; H: espesor
    supuesto). Devuelve (c, e_máx en r_i [% del rango])."""
    ri, Pw = partes(y)
    lam = np.sqrt(ri[:, None] ** 2 + R0 ** 2 - (R0 - H / 2) ** 2) / R0
    B = np.maximum(0.0, lam - (NUDOS + d)[None, :]) / lam ** 2
    c = np.linalg.lstsq(B, Pw, rcond=None)[0]
    Pf = B @ c
    return c, float(np.abs((Pw - Pf) / np.maximum(np.gradient(Pf, ri), 1e-9)).max() / np.ptp(ri) * 100)


def delgada_mejor(y, H=0.5):
    """Búsqueda en grilla de (δ, R0) y refinamiento de δ, para un espesor supuesto H. Devuelve (δ, R0, c)."""
    ri = partes(y)[0]
    mejor = (np.inf, 0.0, ri.mean())
    for R0 in (ri[N_P // 2] + H / 2) / np.linspace(0.95, 1.35, 21):
        for d in np.linspace(-DELTA_MAX, DELTA_MAX, 51):
            e = delgada(y, d, R0, H)[1]
            if e < mejor[0]:
                mejor = (e, d, R0)
    _, d0, R0 = mejor
    d = minimize_scalar(lambda d: delgada(y, d, R0, H)[1], bounds=(max(d0 - 0.01, -DELTA_MAX), min(d0 + 0.01, DELTA_MAX)),
                        method="bounded").x
    return d, R0, delgada(y, d, R0, H)[0]


print(f"Materiales (capa 1 = junto a la sangre) | δ en ±{DELTA_MAX:g} | R_g en {R_G_LIM[0]:g}–{R_G_LIM[1]:g} mm:")
print(pd.DataFrame({"material": np.arange(1, N_BASES + 1), "nudo k": NUDOS, "E [kPa]": E_MAT.round(1)}).to_string(index=False))

## Celda 4 — Ajuste por mínimos cuadrados no lineales de los 41 sujetos

In [ ]:
# CELDA 4: AJUSTE EXACTO (referencia) — mínimos cuadrados NO lineales del diseño θ = (h_1 … h_N, δ, R_g)
#   min Σ_k (r_i,injerto(P_w,k; θ) − r_i,arteria(P_i,k))² en la ventana de presión (región de confianza reflectiva «trf»,
#   scipy least_squares). Arranque: pared delgada (δ y R0 por búsqueda en grilla; h_j = c_j·R0/E_j; R_g = R0 − H/2) o el que se le dé
#   (p. ej. la red). Límites solo físicos: 0 ≤ h_j ≤ H_CAPA_MAX, |δ| ≤ DELTA_MAX, R_g en R_G_LIM.
import time
import numpy as np
from scipy.optimize import least_squares

LO = np.r_[np.zeros(N_BASES), -DELTA_MAX, R_G_LIM[0]]
HI = np.r_[np.full(N_BASES, H_CAPA_MAX), DELTA_MAX, R_G_LIM[1]]


def arranque_delgada(y):
    """MCO lineal de pared delgada: primero con H = 0,5 mm y luego con el espesor H que resultó (3 vueltas). R_g = R0 − H/2."""
    H = 0.5
    for _ in range(3):
        d, R0, c = delgada_mejor(y, H)
        h = np.clip(c * R0 / E_MAT, 0.0, H_CAPA_MAX)
        H = float(np.clip(h.sum(), 0.1, 2.0))
    return np.r_[h, d, np.clip(R0 - h.sum() / 2, *R_G_LIM)]


def ajuste_exacto(y, th0=None, info=False):
    ri, Pw = partes(y)
    th0 = arranque_delgada(y) if th0 is None else np.asarray(th0, float)
    th0 = np.clip(th0, LO + 1e-4, HI - 1e-4)
    r = least_squares(lambda th: (curva_injerto(th, Pw) - ri) / np.ptp(ri), th0, bounds=(LO, HI), method="trf",
                      x_scale=np.r_[np.maximum(th0[:N_BASES], 0.02), 0.01, 0.05], xtol=1e-10, ftol=1e-10, max_nfev=300)
    return (r.x, r.nfev) if info else r.x


t0 = time.perf_counter()
_R = [ajuste_exacto(y, info=True) for y in POB_Y]
T_EXACTO = (time.perf_counter() - t0) / len(POB_Y)
TH_SUJ = np.array([r[0] for r in _R])
E_SUJ = np.array([errores(y, curva_de(th, y)) for y, th in zip(POB_Y, TH_SUJ)])
print(f"✅ Ajuste exacto de los 41 sujetos ({1e3 * T_EXACTO:.0f} ms y {np.mean([r[1] for r in _R]):.0f} evaluaciones por sujeto): "
      f"error máx. en la luz mediana {np.median(E_SUJ[:, 1]):.2f} %, peor {E_SUJ[:, 1].max():.2f} % ({POB.id[E_SUJ[:, 1].argmax()]}) | "
      f"≤ {ERROR_MAX_PCT:g} %: {(E_SUJ[:, 1] <= ERROR_MAX_PCT).sum()}/41 | pared: {TH_SUJ[:, :N_BASES].sum(1).min():.2f}–"
      f"{TH_SUJ[:, :N_BASES].sum(1).max():.2f} mm | δ: {TH_SUJ[:, N_BASES].min():.3f}…{TH_SUJ[:, N_BASES].max():.3f} | "
      f"R_g: {TH_SUJ[:, -1].min():.2f}–{TH_SUJ[:, -1].max():.2f} mm")

## Celda 5 — Dataset con etiquetas

In [ ]:
# CELDA 5: DATASET CON ETIQUETAS (diseño exacto θ = h_1 … h_N, δ, R_g)
#   Familia 1 (injertos): se sortea el diseño θ y la presión perivascular de una arteria al azar; la curva es la luz del injerto
#   bajo esa carga → la etiqueta es exactamente θ.
#   Familias 2, 3, 5 y 6 (arterias): la etiqueta es el ajuste exacto (Celda 4) de la curva. Es lento (≈ 0,7 s por curva):
#   se guarda en ARCHIVO_ETIQUETAS (con avance parcial cada 500 curvas) y las veces siguientes se carga.
import os
import time
import numpy as np

N_INJERTOS = 6000
rng_b = np.random.default_rng(2028)
H_GEN_MAX = 1.3 * TH_SUJ[MASK_TRAIN, :N_BASES].max(0)
D_GEN = (TH_SUJ[MASK_TRAIN, N_BASES].min() - 0.03, TH_SUJ[MASK_TRAIN, N_BASES].max() + 0.03)
R_GEN = (0.9 * TH_SUJ[MASK_TRAIN, -1].min(), 1.1 * TH_SUJ[MASK_TRAIN, -1].max())
_PW_POOL = np.vstack([W_ESP[:, N_P:], POB_PW[MASK_TRAIN]])


def _sortear_injerto():
    th = np.zeros(N_BASES + 2)
    act = rng_b.choice(N_BASES, size=rng_b.integers(2, N_BASES + 1), replace=False)
    th[act] = rng_b.uniform(0.0, 1.0, size=len(act)) * H_GEN_MAX[act]
    th[N_BASES] = np.clip(rng_b.uniform(*D_GEN), -DELTA_MAX, DELTA_MAX)
    th[-1] = rng_b.uniform(*R_GEN)
    return th


if os.path.exists(ARCHIVO_ETIQUETAS):
    _d = np.load(ARCHIVO_ETIQUETAS)
    TH_INJ, Y_INJ, TH_ART = _d["TH_INJ"], _d["Y_INJ"], _d["TH_ART"]
    print(f"📦 Etiquetas cargadas de '{ARCHIVO_ETIQUETAS}'")
else:
    t0 = time.time()
    TH_INJ, Y_INJ = [], []
    while len(TH_INJ) < N_INJERTOS:                                     # injertos que cubren la ventana con forma razonable
        th = _sortear_injerto()
        Pw = _PW_POOL[rng_b.integers(len(_PW_POOL))]
        B, ri = aportes(th[:N_BASES], th[N_BASES], th[-1], X_INJ)
        P = np.maximum.accumulate(B @ th[:N_BASES])
        if P[0] < Pw[0] and P[-1] > Pw[-1]:
            l = curva_injerto(th, Pw)
            if np.ptp(l) > 0.02 * l.mean():
                TH_INJ.append(th)
                Y_INJ.append(np.r_[l, Pw])
    TH_INJ, Y_INJ = np.array(TH_INJ), np.array(Y_INJ)
    _Y = np.vstack([LIMPIAS[f] for f in (2, 3, 5)] + [POB_Y[MASK_TRAIN]])
    TH_ART, i0, _parcial = np.zeros((len(_Y), N_BASES + 2)), 0, ARCHIVO_ETIQUETAS.replace(".npz", "_parcial.npz")
    if os.path.exists(_parcial):
        _p = np.load(_parcial)
        i0 = int(_p["n"])
        TH_ART[:i0] = _p["TH_ART"][:i0]
        print(f"   continúa desde la curva {i0}", flush=True)
    print(f"⏳ Ajuste exacto de {len(_Y)} arterias (≈ {0.7 * (len(_Y) - i0) / 60:.0f} min)...", flush=True)
    for i in range(i0, len(_Y)):
        TH_ART[i] = ajuste_exacto(_Y[i])
        if (i + 1) % 500 == 0:
            np.savez(_parcial, TH_ART=TH_ART, n=i + 1)
            print(f"   {i + 1}/{len(_Y)} | {(time.time() - t0) / 60:.1f} min", flush=True)
    np.savez_compressed(ARCHIVO_ETIQUETAS, TH_INJ=TH_INJ, Y_INJ=Y_INJ, TH_ART=TH_ART)
    if os.path.exists(_parcial):
        os.remove(_parcial)
    print(f"✅ Etiquetas en {(time.time() - t0) / 60:.1f} min → {ARCHIVO_ETIQUETAS}")

_n6 = MASK_TRAIN.sum()
Y_ALL = np.vstack([Y_INJ] + [LIMPIAS[f] for f in (2, 3, 5)] + [POB_Y[MASK_TRAIN][np.resize(np.arange(_n6), N_FAM[6])]])
TH_ALL = np.vstack([TH_INJ, TH_ART[:-_n6], TH_ART[-_n6:][np.resize(np.arange(_n6), N_FAM[6])]])
familia = np.concatenate([np.full(len(Y_INJ), 1)] + [np.full(N_FAM[f], f) for f in (2, 3, 5, 6)]).astype(np.int8)
_E = np.array([errores(y, curva_de(t, y))[1] for y, t in zip(Y_ALL[familia != 1][::20], TH_ALL[familia != 1][::20])])
print(f"✅ Dataset: {len(Y_ALL)} curvas ({len(Y_INJ)} injertos + {len(Y_ALL) - len(Y_INJ)} arterias) | etiqueta exacta de las arterias: "
      f"error máx. en la luz mediana {np.median(_E):.2f} %, p95 {np.percentile(_E, 95):.2f} % (muestra de {len(_E)})")

## Celda 6 — Partición y normalización

In [ ]:
# CELDA 6: PARTICIÓN TRAIN / VAL / TEST Y NORMALIZACIÓN
#   Entrada de la red: la curva curva [r_i(P_i) ; P_i − P_a] en los N_P valores de presión (2·N_P entradas), estandarizada. Salida: el diseño th estandarizado.
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

idx_trval, idx_test = train_test_split(np.arange(len(Y_ALL)), test_size=0.15, random_state=SEED, stratify=familia)
idx_train, idx_val = train_test_split(idx_trval, test_size=0.12, random_state=SEED, stratify=familia[idx_trval])
sx, sy = StandardScaler().fit(Y_ALL[idx_train]), StandardScaler().fit(TH_ALL[idx_train])
X_train, X_val, X_test = (sx.transform(Y_ALL[i]).astype(np.float32) for i in (idx_train, idx_val, idx_test))
y_train, y_val, y_test = (sy.transform(TH_ALL[i]).astype(np.float32) for i in (idx_train, idx_val, idx_test))


def predecir(Y, red=None):
    red = model if red is None else red
    th = sy.inverse_transform(red.predict(sx.transform(np.atleast_2d(Y)), batch_size=1024, verbose=0))
    return th[0] if np.ndim(Y) == 1 else th


print(f"✅ Train {len(idx_train)} | Val {len(idx_val)} | Test {len(idx_test)}")

## Celda 7 — Red: entrenamiento (o carga)

In [ ]:
# CELDA 7: RED Y ENTRENAMIENTO (o carga)
#   Red 282 → 256 → 128 → 64 → N+2 (ReLU + LayerNormalization, salida lineal). Entrada: la luz r_i y la carga de pared P_i − P_a
#   en los 141 valores de P_i. Salida: θ = (h_1 … h_N, δ, R_g). Pérdida: error cuadrático medio del diseño estandarizado.
#   En cada época, a la luz se le suma un ruido nuevo (0 a 1 % de su rango en cada curva) para que la red no dependa de curvas perfectas.
import os, time, shutil, pickle
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import callbacks as cb
from tensorflow.keras.models import load_model

tf.random.set_seed(SEED)
EPOCHS, BATCH_SIZE, RUIDO_MAX = 1500, 256, 0.01
_sxm, _sxs = tf.constant(sx.mean_, tf.float32), tf.constant(sx.scale_, tf.float32)


def construir_red():
    capas = [tf.keras.Input(shape=(2 * N_P,))]
    for u in (256, 128, 64):
        capas += [tf.keras.layers.Dense(u, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
                  tf.keras.layers.LayerNormalization()]
    return tf.keras.Sequential(capas + [tf.keras.layers.Dense(N_BASES + 2)])


def _con_ruido(x, y):
    v = x * _sxs + _sxm
    l, pw = v[:, :N_P], v[:, N_P:]
    rango = tf.reduce_max(l, axis=1, keepdims=True) - tf.reduce_min(l, axis=1, keepdims=True)
    u = tf.random.uniform([tf.shape(x)[0], 1], 0.0, RUIDO_MAX)
    v = tf.concat([l + tf.random.normal(tf.shape(l)) * u * rango, pw], axis=1)
    return (v - _sxm) / _sxs, y


if ENTRENAR:
    model = construir_red()
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0), loss="mse")
    t0, _resp, _log = time.time(), f"_respaldo_{VERSION}", f"_registro_{VERSION}.csv"
    ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(len(X_train), seed=SEED, reshuffle_each_iteration=True)
          .batch(BATCH_SIZE).map(_con_ruido, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE))
    model.fit(ds, validation_data=(X_val, y_val), epochs=EPOCHS, verbose=2,
              callbacks=[cb.TerminateOnNaN(), cb.EarlyStopping(patience=120, min_delta=1e-6, restore_best_weights=True),
                         cb.ReduceLROnPlateau(factor=0.5, patience=40, min_delta=1e-6, min_lr=1e-6),
                         cb.BackupAndRestore(_resp), cb.CSVLogger(_log, append=True)])
    hist = pd.read_csv(_log).drop_duplicates("epoch", keep="last").drop(columns="epoch")
    hist.index = pd.RangeIndex(1, len(hist) + 1, name="epoca")
    shutil.rmtree(_resp, ignore_errors=True)
    os.remove(_log)
    model.save(ARCHIVO_MODELO)
    with open(ARCHIVO_SCALERS, "wb") as f:
        pickle.dump(dict(scaler_x=sx, scaler_y=sy, E_MAT=E_MAT, NUDOS=NUDOS), f)
    hist.to_csv(ARCHIVO_HISTORIAL)
    print(f"✅ Entrenada en {(time.time() - t0) / 60:.1f} min ({len(hist)} épocas)")
else:
    model = load_model(ARCHIVO_MODELO, compile=False)
    hist = pd.read_csv(ARCHIVO_HISTORIAL, index_col="epoca")
    with open(ARCHIVO_SCALERS, "rb") as f:
        _m = pickle.load(f)
    assert np.allclose(_m["E_MAT"], E_MAT) and np.allclose(_m["NUDOS"], NUDOS), "la red se entrenó con otros materiales"
    sx, sy = _m["scaler_x"], _m["scaler_y"]
    print(f"📦 Red cargada de '{ARCHIVO_MODELO}' ({len(hist)} épocas)")
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.semilogy(hist.index, hist["loss"], label="Entrenamiento")
ax.semilogy(hist.index, hist["val_loss"], "--", label="Validación")
ax.set(xlabel="Época", ylabel="Pérdida", title=f"Red {VERSION}")
ax.legend()
ax.grid(alpha=0.3)
plt.savefig(f"evolucion_perdida_{VERSION}.png", dpi=180, bbox_inches="tight")
plt.show()

## Celda 8 — Dos caminos al diseño entero: arranque MCO lineal y arranque red (curvas limpias)

In [ ]:
# CELDA 8: DOS CAMINOS AL DISEÑO ENTERO, con el MISMO procedimiento de 3 pasos; solo cambia el arranque
#   Paso 1 · arranque ......... A) MCO lineal de pared delgada (arranque MCO lineal)   B) red neuronal (arranque red)
#   Paso 2 · MCO no lineal ..... mínimos cuadrados no lineales de pared gruesa (números reales): ajusta la luz y, si la curva viene
#                                medida con ruido o en pocas presiones, la suaviza (la curva de un injerto no puede tener saltos)
#   Paso 3 · láminas enteras ... minimax no lineal contra la curva del paso 2: combinaciones de láminas de ESP_LAMINA_MM alrededor
#                                del diseño del paso 2, con δ y R_g reajustados y mejora local; gana la de peor punto más chico
#   El diseño ENTERO se compara siempre con la curva ORIGINAL del sujeto (la luz a la misma presión de la sangre).
import time
import itertools
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar


def e_max_th(y, th):
    """Peor error de la luz [% del rango de r_i] del diseño th contra la curva y."""
    ri, Pw = partes(y)
    e = float(np.abs(curva_injerto(th, Pw) - ri).max() / np.ptp(ri) * 100)
    return e if np.isfinite(e) else np.inf


def a_enteros(y, th, esp=None, n_mejores=8):
    """Diseño con láminas enteras (n_j ≥ 0) de espesor esp, con δ y R_g reajustados, por el criterio del peor punto. Devuelve (θ, n)."""
    esp = ESP_LAMINA_MM if esp is None else esp
    n0 = np.maximum(th[:N_BASES], 0.0) / esp
    opciones = [sorted({int(np.floor(v)), int(np.ceil(v))}) for v in n0]
    d0, R0 = th[N_BASES], th[-1]
    cand = sorted(((e_max_th(y, np.r_[np.asarray(n, float) * esp, d0, R0]), np.asarray(n, float)) for n in itertools.product(*opciones)),
                  key=lambda c: c[0])
    mejor = (cand[0][0], d0, R0, cand[0][1])

    def _fino(n, d, R):
        """δ (±0,03), luego R_g (±3 %), luego δ otra vez: búsquedas de una variable acotadas (Brent)."""
        rd = lambda d, R: minimize_scalar(lambda v: e_max_th(y, np.r_[n * esp, v, R]), method="bounded", options=dict(xatol=1e-5),
                                          bounds=(max(d - 0.03, -DELTA_MAX), min(d + 0.03, DELTA_MAX))).x
        d = rd(d, R)
        R = minimize_scalar(lambda v: e_max_th(y, np.r_[n * esp, d, v]), method="bounded", options=dict(xatol=1e-5),
                            bounds=(max(0.97 * R, R_G_LIM[0]), min(1.03 * R, R_G_LIM[1]))).x
        d = rd(d, R)
        return d, R, e_max_th(y, np.r_[n * esp, d, R])

    for _, n in cand[:n_mejores]:
        d, R, e = _fino(n, d0, R0)
        if e < mejor[0]:
            mejor = (e, d, R, n)
    e, d, R, n = mejor
    for _ in range(20):                                  # mejora local: mover una lámina mientras baje el peor error
        vecinos = [n + s * np.eye(N_BASES)[j] for j in range(N_BASES) for s in (-1, 1)]
        vecinos += [n + np.eye(N_BASES)[i] - np.eye(N_BASES)[j] for i in range(N_BASES) for j in range(N_BASES) if i != j]
        vecinos = [v for v in vecinos if v.min() >= 0]
        pre = sorted(((e_max_th(y, np.r_[v * esp, d, R]), i) for i, v in enumerate(vecinos)))[:n_mejores]
        dd, RR, ee, nn = min(((*_fino(vecinos[i], d, R), vecinos[i]) for _, i in pre), key=lambda c: c[2])
        if ee < e - 1e-6:
            e, d, R, n = ee, dd, RR, nn
        else:
            break
    return np.r_[n * esp, d, R], n.astype(int)


def red(y):
    return sy.inverse_transform(model(sx.transform(np.atleast_2d(y)).astype("float32")).numpy())[0]


CAMINOS = ["A) Arranque MCO lineal", "B) Arranque red"]


def procedimiento(y, th0):
    """Pasos 2 y 3 desde el arranque th0: (θ entero, n láminas, θ del MCO no lineal)."""
    th_mc = ajuste_exacto(y, th0=th0)                                         # paso 2 (reales)
    th, n = a_enteros(np.r_[curva_de(th_mc, y), partes(y)[1]], th_mc)        # paso 3 (enteros, minimax contra la curva del ajuste)
    return th, n, th_mc


def caminos(y):
    """Diseño entero de cada camino para la curva y: {camino: (θ, n, tiempo s)}."""
    out = {}
    for m, arranque in zip(CAMINOS, (arranque_delgada, red)):
        t0 = time.perf_counter()
        th, n, _ = procedimiento(y, arranque(y))
        out[m] = (th, n, time.perf_counter() - t0)
    return out


model(sx.transform(POB_Y[:2]).astype("float32"))
t00 = time.time()
_fil, DIS_SUJ = [], {}
for nm, y, es_test in zip(POB.id, POB_Y, MASK_TEST):
    D = caminos(y)
    DIS_SUJ[nm] = D
    for m, (th, n, t) in D.items():
        _fil.append({"sujeto": nm, "grupo": "test (10, la red nunca los vio)" if es_test else "entrenamiento (31)", "camino": m,
                     "peor error %": e_max_th(y, th), "tiempo s": t, "láminas": int(n.sum()), "pared mm": th[:N_BASES].sum(),
                     "delta": th[N_BASES], "R_g mm": th[-1], **{f"n{j + 1}": int(n[j]) for j in range(N_BASES)}})
COMP = pd.DataFrame(_fil)
COMP.to_csv(f"comparacion_{VERSION}.csv", index=False)
print(f"Curvas LIMPIAS · diseño ENTERO (láminas de {ESP_LAMINA_MM:g} mm) contra la curva ORIGINAL — peor error de la luz [% del rango] | "
      f"41 sujetos | {(time.time() - t00) / 60:.1f} min")
_res = COMP.groupby("camino", sort=False).agg(
    mediana=("peor error %", "median"), peor=("peor error %", "max"), dentro_3=("peor error %", lambda v: f"{(v <= 3).sum()}/{len(v)}"),
    tiempo_s=("tiempo s", "median"), láminas=("láminas", "median"), pared_mm=("pared mm", "median"), R_g_mm=("R_g mm", "median"))
print(_res.round(2).to_string())
print("\nSolo los 10 sujetos de test (la red nunca los vio): mediana / peor")
print(COMP[COMP.grupo.str.startswith("test")].groupby("camino", sort=False)["peor error %"].agg(["median", "max"]).round(2).to_string())
_res.round(3).to_csv(f"resumen_{VERSION}.csv")
_g = COMP.pivot(index="sujeto", columns="camino", values="peor error %")
print("\nArranque red mejor que arranque MCO lineal en", int((_g[CAMINOS[1]] < _g[CAMINOS[0]] - 1e-6).sum()), "sujetos, peor en",
      int((_g[CAMINOS[1]] > _g[CAMINOS[0]] + 1e-6).sum()), ", igual en el resto")

## Celda 9 — Figuras en las dos escalas

In [ ]:
# CELDA 9: FIGURAS — tres sujetos (el más distensible, uno intermedio y el más rígido): curva original y los dos diseños ENTEROS
#   Abscisa: diámetro de la luz D_i = 2·r_i [mm]. Ordenada: presión de la sangre P_i (arriba en kPa, abajo P_i/P_ref con
#   P_ref = 13,332 kPa); ordenada derecha en mmHg. Punteada gris: lo que soporta la pared de la arteria, P_i − P_a.
#   Cada injerto se dibuja a la presión de la sangre P_i que le corresponde (su pared soporta P_i − P_a, la misma P_a que la arteria).
import numpy as np
import matplotlib.pyplot as plt

_dist = (POB_RI[:, -1] - POB_RI[:, 0]) / POB_RI[:, 0]
_ord = np.argsort(_dist)
_tres = [_ord[-1], _ord[len(_ord) // 2], _ord[0]]
_est = dict(zip(CAMINOS, (("tab:blue", "--"), ("tab:red", "-"))))
fig, axs = plt.subplots(2, 3, figsize=(18, 10))
for col, i in enumerate(_tres):
    y, D = POB_Y[i], DIS_SUJ[POB.id[i]]
    for fila, esc in ((0, 1.0), (1, P_REF_KPA)):
        ax = axs[fila, col]
        ax.plot(2 * POB_RI[i], P_GRID / esc, "k-", lw=3, label="Arteria: P_i (curva original)")
        ax.plot(2 * POB_RI[i], POB_PW[i] / esc, color="0.5", ls=":", lw=1.5, label="Arteria: P_i − P_a (lo que soporta la pared)")
        for m, (th, n, _) in D.items():
            c, ls = _est[m]
            ax.plot(2 * curva_de(th, y), P_GRID / esc, color=c, ls=ls, lw=1.8,
                    label=f"{m}: {n.tolist()}, R_g = {th[-1]:.2f} mm (peor {e_max_th(y, th):.2f} %)")
        eje_mmhg(ax, esc)
        ax.set(xlabel=ETQ_D, ylabel=ETQ_P if fila == 0 else f"P / P_ref  (P_ref = {P_REF_KPA:.3f} kPa)",
               title=f"{POB.id[i]} (distensión 60→200 mmHg: {100 * _dist[i]:.1f} %)")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=7.5, loc="upper left")
fig.suptitle(f"{N_BASES} materiales · se copia el diámetro de la luz · láminas de {ESP_LAMINA_MM:g} mm (entre corchetes, láminas de cada material)")
fig.tight_layout()
plt.savefig(f"disenos_{VERSION}.png", dpi=180, bbox_inches="tight")
plt.show()

# peor error de cada sujeto, por camino
_g = COMP.pivot(index="sujeto", columns="camino", values="peor error %")[CAMINOS].loc[list(POB.id)]
fig, ax = plt.subplots(figsize=(18, 5))
x = np.arange(len(_g))
for k, m in enumerate(CAMINOS):
    ax.bar(x + (k - 0.5) * 0.4, _g[m], width=0.4, color=_est[m][0], label=m)
ax.axhline(ERROR_MAX_PCT, color="k", ls="--", lw=1.2, label=f"meta {ERROR_MAX_PCT:g} %")
ax.set_xticks(x, _g.index, rotation=90, fontsize=8)
ax.set(ylabel="peor error de la luz contra la curva original [%]", title=f"{N_BASES} materiales · peor error de cada sujeto con láminas enteras")
ax.legend()
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
plt.savefig(f"peor_error_por_sujeto_{VERSION}.png", dpi=180, bbox_inches="tight")
plt.show()

## Celda 10 — Curvas medidas: ¿la red evita diseños que fallan?

In [ ]:
# CELDA 10: ¿JUSTIFICA LA RED? — los mismos dos caminos con curvas MEDIDAS (ruido en el diámetro o pocas presiones)
#   En la práctica la luz del paciente se mide (ecografía), con ruido y en pocas presiones. El ruido se pone en el diámetro; la
#   carga de pared P_i − P_a se toma como conocida. Cada camino recibe la curva MEDIDA y su diseño entero se compara con la curva
#   VERDADERA (la original, sin ruido). Una realización por sujeto y escenario.
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator

ESC_ROB = {"ruido 0,3 %": (0.003, None), "10 presiones": (0.0, 10), "10 presiones + ruido 0,2 %": (0.002, 10)}
_rng = np.random.default_rng(SEED + 5)
_fil = []
t00 = time.time()
for esc, (r, npts) in ESC_ROB.items():
    for nm, y in zip(POB.id, POB_Y):
        ri, Pw = partes(y)
        if npts:
            pm = np.linspace(P_GRID[0], P_GRID[-1], npts)
            rim = np.interp(pm, P_GRID, ri)
            rim = PchipInterpolator(pm, rim + _rng.normal(0.0, r, size=npts) * rim)(P_GRID)
            Pwm = PchipInterpolator(pm, np.interp(pm, P_GRID, Pw))(P_GRID)
        else:
            rim, Pwm = ri + _rng.normal(0.0, r, size=ri.shape) * ri, Pw
        for m, (th, n, t) in caminos(np.r_[rim, Pwm]).items():
            _fil.append({"escenario": esc, "camino": m, "sujeto": nm, "peor error %": e_max_th(y, th), "tiempo s": t})
ROB = pd.DataFrame(_fil)
ROB.to_csv(f"robustez_{VERSION}.csv", index=False)
print(f"Diseño ENTERO a partir de la curva MEDIDA, comparado con la curva VERDADERA — peor error de la luz [%] | 41 sujetos | "
      f"{(time.time() - t00) / 60:.1f} min")
print(ROB.groupby(["escenario", "camino"], sort=False).agg(
    mediana=("peor error %", "median"), peor=("peor error %", "max"), dentro_3=("peor error %", lambda v: f"{(v <= 3).sum()}/{len(v)}"),
    fallan_10=("peor error %", lambda v: int((v > 10).sum())), tiempo_s=("tiempo s", "median")).round(2).to_string())

fig, axs = plt.subplots(len(ESC_ROB), 1, figsize=(18, 4.2 * len(ESC_ROB)))
for ax, esc in zip(axs, ESC_ROB):
    g = ROB[ROB.escenario == esc].pivot(index="sujeto", columns="camino", values="peor error %").loc[list(POB.id)][CAMINOS]
    x = np.arange(len(g))
    for k, m in enumerate(CAMINOS):
        ax.bar(x + (k - 0.5) * 0.4, g[m], width=0.4, color=("tab:blue", "tab:red")[k],
               label=f"{m}: {int((g[m] <= ERROR_MAX_PCT).sum())}/41 dentro de la meta, peor {g[m].max():.1f} %")
    ax.set_yscale("log"); ax.axhline(ERROR_MAX_PCT, color="k", ls="--", label=f"meta {ERROR_MAX_PCT:g} %")
    ax.set_xticks(x, g.index, rotation=90, fontsize=8); ax.set_ylabel("peor error [%] (log)")
    ax.set_title(f"{N_BASES} materiales · curva medida: {esc} · diseño entero contra la curva VERDADERA")
    ax.legend(fontsize=9, loc="upper right"); ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
plt.savefig(f"curvas_medidas_{VERSION}.png", dpi=150, bbox_inches="tight")
plt.show()